[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_11_LangGraph_Stateful_Agent_Graphs.ipynb)

# 🧠 Lesson 11: LangGraph — Stateful Agent Graphs

**Phase 2 | Lesson 11 of 15 | AI/LLM/Agents Curriculum**

---

## What You'll Learn Today

So far you've built agents by hand — writing ReAct loops, tool dispatchers, and memory managers in plain Python. That works, but it gets messy fast once you need:
- **Branching logic** (go left if tool was called, go right if done)
- **Cycles** (loop back to the LLM after a tool runs)
- **Persistence** (pause mid-graph, resume later with memory intact)
- **Parallel branches** (fan-out to multiple subagents simultaneously)

**LangGraph** is a library built on top of LangChain that models agent behaviour as a **directed graph** (nodes + edges). It's the tool of choice for production-grade multi-step agents at companies like LinkedIn, Elastic, and Replit.

Today's agenda:
1. Core concepts — State, Nodes, Edges, Graph
2. Hello World graph
3. ReAct agent the LangGraph way (tools + cycles)
4. Conditional branching
5. Checkpointing (pause & resume)
6. Capstone: A decision-making research agent

> **Prerequisites:** Lessons 1–10. Especially Lesson 4 (ReAct loop) and Lesson 3 (Tool Use).

## 🔧 Setup — Install & Authenticate

Run this cell first. It installs everything and loads your Anthropic API key from Colab Secrets.

> **One-time setup:** In Colab, click the 🔑 **Secrets** tab (left sidebar), add a secret named `ANTHROPIC_API_KEY` with your key from https://console.anthropic.com

In [ ]:
# Install required packages
!pip install langgraph langchain-anthropic langchain-core anthropic -q

import os
import json
from typing import TypedDict, Annotated, Literal
from IPython.display import display, Markdown

# Load API key from Colab Secrets (or set directly for local use)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Running locally — set your key here
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-YOUR-KEY-HERE"
    print("⚠️  Set your ANTHROPIC_API_KEY above for local use")

print("✅ All packages ready — let's build graphs!")

---

## 🧩 Part 1: The Core Mental Model

### Why graphs for agents?

Think about what a ReAct agent actually does:

```
start → call_llm → (did it call a tool?) → YES → run_tool → call_llm → ...
                                         → NO  → END
```

That's a graph! A directed graph with nodes (functions) and edges (routing logic). LangGraph makes this explicit instead of burying it in `while` loops and `if` statements.

### The 4 key building blocks

| Concept | What it is | Analogy |
|---------|-----------|--------|
| **State** | A Python dict (TypedDict) that flows through the entire graph | The baton in a relay race — everyone can read/update it |
| **Node** | A Python function that receives State, does work, returns State updates | A worker at a station in an assembly line |
| **Edge** | A connection from one node to another | A conveyor belt between stations |
| **Conditional Edge** | A function that *decides* which node to go to next | A traffic light / switch |

### State is the key insight

Every node reads from and writes to a **shared State object**. Nodes don't pass data to each other directly — they update the State, and LangGraph passes it along. This is what makes the graph stateful and what enables checkpointing (saving state to disk mid-run).

```python
# State is a TypedDict — a dictionary with typed fields
class MyState(TypedDict):
    messages: list      # conversation history
    step_count: int     # how many steps we've taken
    final_answer: str   # where we store the result
```

LangGraph uses **reducers** to handle how state updates are merged. By default, a field is overwritten. But for `messages`, you typically use `add_messages` — an append reducer, so messages accumulate rather than getting replaced.

---

## 🌍 Part 2: Hello World — Your First Graph

Let's build the simplest possible LangGraph: one node that calls Claude, then ends.

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# ── Step 1: Define State ──────────────────────────────────────────────────────
# Annotated[list, add_messages] means: this list uses the add_messages reducer
# (new messages are APPENDED, not overwritten)
class BasicState(TypedDict):
    messages: Annotated[list, add_messages]

# ── Step 2: Define the LLM ────────────────────────────────────────────────────
llm = ChatAnthropic(
    model="claude-haiku-4-5",   # Fast & cheap for learning
    max_tokens=500
)

# ── Step 3: Define a Node ─────────────────────────────────────────────────────
# A node is just a function: State -> dict of updates
def call_llm(state: BasicState) -> dict:
    """Call Claude with the current message history."""
    print("🤖 [call_llm node] Calling Claude...")
    response = llm.invoke(state["messages"])
    return {"messages": [response]}  # add_messages will APPEND this

# ── Step 4: Build the Graph ───────────────────────────────────────────────────
builder = StateGraph(BasicState)

# Add nodes
builder.add_node("call_llm", call_llm)

# Set entry point and edges
builder.set_entry_point("call_llm")   # first node to run
builder.add_edge("call_llm", END)      # after call_llm, we're done

# Compile into a runnable graph
graph = builder.compile()

print("✅ Graph built!")
print("Graph nodes:", list(graph.nodes))

In [ ]:
# ── Step 5: Invoke the Graph ──────────────────────────────────────────────────
# We pass in the initial state — just one human message
initial_state = {
    "messages": [HumanMessage(content="What is LangGraph in one sentence?")]
}

result = graph.invoke(initial_state)

print("\n📨 Final message history:")
for msg in result["messages"]:
    role = "👤 Human" if isinstance(msg, HumanMessage) else "🤖 Claude"
    print(f"{role}: {msg.content}")

# 💡 EXPERIMENT: Change the question, add a SystemMessage as the first message,
#    or increase max_tokens to see longer answers.

### 🔍 Visualising the graph flow

LangGraph can stream intermediate state updates as they happen — useful for debugging. Let's see that:

In [ ]:
print("🌊 Streaming graph events:\n")

for event in graph.stream(
    {"messages": [HumanMessage(content="Name three uses of AI agents.")]},
    stream_mode="updates"  # stream state updates after each node
):
    # event is a dict: {node_name: state_updates}
    for node_name, updates in event.items():
        print(f"📍 Node: {node_name}")
        if "messages" in updates:
            last_msg = updates["messages"][-1]
            print(f"   └── Output: {last_msg.content[:150]}...")

# 💡 EXPERIMENT: Try stream_mode="values" to see the full state after each node,
#    rather than just the delta/updates.

---

## 🛠️ Part 3: ReAct Agent with Tools (Cycles in LangGraph)

The real power of LangGraph shows up when you add **cycles** — loops where the LLM calls a tool, the tool runs, and then the LLM sees the result and decides whether to call another tool or finish.

This is the ReAct pattern you built manually in Lesson 4, but now the graph structure makes it explicit and controllable.

### The pattern:

```
START → [llm_node] → tool was called? → YES → [tool_node] → back to [llm_node]
                                       → NO  → END
```

The `tool_node` → `llm_node` back-arrow is the **cycle**. LangGraph handles this safely — it won't loop infinitely because Claude eventually stops calling tools.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

# ── Define Tools ──────────────────────────────────────────────────────────────
# LangChain's @tool decorator is the cleanest way to define tools for LangGraph
# It automatically generates the tool schema from the docstring + type hints

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input: a valid Python math expression like '2 ** 10' or '(3 + 5) * 7'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city (simulated)."""
    weather_data = {
        "london": "15°C, cloudy with light rain",
        "tokyo": "22°C, sunny",
        "new york": "18°C, partly cloudy",
        "mumbai": "32°C, humid and hazy",
    }
    return weather_data.get(city.lower(), f"No data available for {city}")

@tool
def lookup_fact(topic: str) -> str:
    """Look up a quick fact about a topic (simulated knowledge base)."""
    facts = {
        "langgraph": "LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of LangChain.",
        "python": "Python was created by Guido van Rossum and first released in 1991.",
        "anthropic": "Anthropic is an AI safety company founded in 2021, creator of the Claude family of AI models.",
    }
    for key in facts:
        if key in topic.lower():
            return facts[key]
    return f"No specific fact found for '{topic}'. General knowledge applies."

tools = [calculate, get_weather, lookup_fact]

print("🔧 Tools registered:")
for t in tools:
    print(f"  • {t.name}: {t.description[:60]}...")

In [ ]:
# ── Build the ReAct Graph ─────────────────────────────────────────────────────

# State: same as before — a list of messages with the add_messages reducer
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# LLM bound to tools — Claude knows what tools it has available
llm_with_tools = ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=1000
).bind_tools(tools)


# ── Node 1: LLM ───────────────────────────────────────────────────────────────
def agent_node(state: AgentState) -> dict:
    """Call the LLM. It may respond with text OR request a tool call."""
    print("🤖 [agent] Thinking...")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# ── Node 2: Tools ─────────────────────────────────────────────────────────────
# LangGraph's built-in ToolNode handles executing tool calls from Claude's response
tool_node = ToolNode(tools)


# ── Conditional Edge: should we call tools or end? ───────────────────────────
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """Look at the last message — if it has tool_calls, route to tools; else end."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print(f"   ↳ Tool calls detected: {[tc['name'] for tc in last_message.tool_calls]}")
        return "tools"
    print("   ↳ No tool calls — finishing")
    return "end"


# ── Build the Graph ───────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

# Entry point
builder.set_entry_point("agent")

# Conditional edge from agent: go to tools OR end
builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "end": END}
)

# After tools run, ALWAYS go back to agent (the cycle!)
builder.add_edge("tools", "agent")

react_graph = builder.compile()
print("✅ ReAct graph compiled!")
print("Graph nodes:", list(react_graph.nodes))

In [ ]:
# ── Run the ReAct Agent ───────────────────────────────────────────────────────
print("═" * 60)
print("QUERY: Multi-step question requiring multiple tools")
print("═" * 60)

query = (
    "I need three things: (1) what is 2 to the power of 15? "
    "(2) what's the weather in Tokyo? "
    "(3) what is Anthropic?"
)

result = react_graph.invoke({
    "messages": [
        SystemMessage(content="You are a helpful assistant. Use tools to answer questions accurately."),
        HumanMessage(content=query)
    ]
})

print("\n" + "─" * 60)
print("FULL CONVERSATION TRACE:")
print("─" * 60)
for msg in result["messages"]:
    msg_type = type(msg).__name__
    if msg_type == "HumanMessage":
        print(f"\n👤 Human: {msg.content}")
    elif msg_type == "SystemMessage":
        print(f"\n⚙️  System: {msg.content}")
    elif msg_type == "AIMessage":
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 Tool Call: {tc['name']}({tc['args']})")
        else:
            print(f"\n🤖 Claude: {msg.content}")
    elif msg_type == "ToolMessage":
        print(f"   📦 Result: {msg.content}")

# 💡 EXPERIMENT: Ask a question that needs ONLY one tool, or NO tools at all.
#    Watch how should_continue() routes differently.

---

## 🌿 Part 4: Richer Branching — State-Driven Routing

Conditional edges don't have to just check for tool calls. They can inspect **any field in State** and route accordingly. This is how you build agents that take different paths based on what they discover.

Let's build a **query classifier** graph that:
1. Classifies the question type (math / weather / fact / other)
2. Routes to a specialised handler for that type
3. Merges results into a final answer

This pattern is called a **router** or **dispatcher** — very common in production agents.

In [ ]:
from langchain_core.messages import AIMessage

# ── State: now with extra fields beyond messages ──────────────────────────────
class RouterState(TypedDict):
    messages: Annotated[list, add_messages]
    query_type: str        # "math" | "weather" | "fact" | "general"
    final_answer: str

fast_llm = ChatAnthropic(model="claude-haiku-4-5", max_tokens=100)


# ── Node: classify the query ──────────────────────────────────────────────────
def classify_query(state: RouterState) -> dict:
    """Use Claude to classify the user's question into a category."""
    user_query = state["messages"][-1].content
    prompt = f"""Classify this query into exactly one word: math, weather, fact, or general.
Query: {user_query}
Answer (one word only):"""
    response = fast_llm.invoke([HumanMessage(content=prompt)])
    query_type = response.content.strip().lower().split()[0]
    if query_type not in ["math", "weather", "fact"]:
        query_type = "general"
    print(f"🔍 Classified as: {query_type}")
    return {"query_type": query_type}


# ── Specialist Nodes ──────────────────────────────────────────────────────────
def handle_math(state: RouterState) -> dict:
    """Handle math questions."""
    user_query = state["messages"][0].content  # original question
    # Extract and evaluate the expression
    response = fast_llm.invoke([
        HumanMessage(content=f"Extract only the math expression from this query and evaluate it. Query: {user_query}")
    ])
    return {"final_answer": f"[Math Handler] {response.content}"}

def handle_weather(state: RouterState) -> dict:
    """Handle weather questions."""
    user_query = state["messages"][0].content
    # Simulate: extract city and look it up
    cities = ["london", "tokyo", "new york", "mumbai"]
    found_city = next((c for c in cities if c in user_query.lower()), None)
    if found_city:
        weather = get_weather.invoke({"city": found_city})
        return {"final_answer": f"[Weather Handler] {weather}"}
    return {"final_answer": "[Weather Handler] City not found in my database."}

def handle_fact(state: RouterState) -> dict:
    """Handle fact lookup questions."""
    user_query = state["messages"][0].content
    result = lookup_fact.invoke({"topic": user_query})
    return {"final_answer": f"[Fact Handler] {result}"}

def handle_general(state: RouterState) -> dict:
    """Handle general questions with Claude directly."""
    user_query = state["messages"][0].content
    response = fast_llm.invoke([HumanMessage(content=user_query)])
    return {"final_answer": f"[General Handler] {response.content}"}


# ── Routing function ──────────────────────────────────────────────────────────
def route_by_type(state: RouterState) -> str:
    """Return the name of the next node based on query_type in state."""
    return state["query_type"]  # directly maps to node names


# ── Build the Router Graph ────────────────────────────────────────────────────
builder = StateGraph(RouterState)

builder.add_node("classify", classify_query)
builder.add_node("math", handle_math)
builder.add_node("weather", handle_weather)
builder.add_node("fact", handle_fact)
builder.add_node("general", handle_general)

builder.set_entry_point("classify")

# Conditional edge: after classify, go to whichever specialist matches
builder.add_conditional_edges(
    "classify",
    route_by_type,
    {"math": "math", "weather": "weather", "fact": "fact", "general": "general"}
)

# All specialists end the graph
for node in ["math", "weather", "fact", "general"]:
    builder.add_edge(node, END)

router_graph = builder.compile()
print("✅ Router graph built!")

In [ ]:
# ── Test the Router ───────────────────────────────────────────────────────────
test_queries = [
    "What is 144 divided by 12?",
    "What is the weather like in London?",
    "Tell me about Anthropic.",
    "What is the meaning of life?"
]

for query in test_queries:
    print(f"\n{'═'*55}")
    print(f"❓ Query: {query}")
    result = router_graph.invoke({
        "messages": [HumanMessage(content=query)],
        "query_type": "",
        "final_answer": ""
    })
    print(f"✅ Answer: {result['final_answer']}")

# 💡 EXPERIMENT: Add a new query type (e.g., "code") with its own handler node.
#    Update classify_query to include it and add the new conditional branch.

---

## 💾 Part 5: Checkpointing — Pause, Persist, Resume

This is one of LangGraph's most powerful features: **checkpointing**.

### What is checkpointing?

After each node runs, LangGraph can save the current State to storage (memory, SQLite, Postgres). This means:
- **Pause and resume**: run half the graph now, come back in an hour and continue
- **Multi-turn conversations**: the graph "remembers" what happened in previous turns
- **Human-in-the-loop**: pause mid-graph, let a human approve, then resume
- **Error recovery**: if a node crashes, retry from the last checkpoint

### Thread IDs

Checkpoints are namespaced by **thread_id**. Think of a thread like a conversation session — same thread_id means same conversation.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# ── Rebuild the ReAct graph WITH a checkpointer ───────────────────────────────
# The only change: pass a checkpointer to .compile()

memory = MemorySaver()  # In-memory checkpointer (for dev/testing)
                         # Production: use SqliteSaver or PostgresSaver

# Rebuild the same graph (reusing from Part 3)
builder2 = StateGraph(AgentState)
builder2.add_node("agent", agent_node)
builder2.add_node("tools", tool_node)
builder2.set_entry_point("agent")
builder2.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder2.add_edge("tools", "agent")

# KEY CHANGE: Add checkpointer here
persistent_graph = builder2.compile(checkpointer=memory)

print("✅ Persistent graph compiled with MemorySaver")
print("\nNow we can have multi-turn conversations where the agent remembers context!")

In [ ]:
# ── Multi-turn Conversation via thread_id ──────────────────────────────────────
# The thread_id is the key: same ID = same conversation memory
config = {"configurable": {"thread_id": "conversation-001"}}

def chat(user_message: str, config: dict):
    """Send a message and get a response, with full conversation memory."""
    result = persistent_graph.invoke(
        {"messages": [HumanMessage(content=user_message)]},
        config=config
    )
    # Get the last AI message (may come after tool calls)
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            return msg.content
    return "(no text response)"

# Turn 1
print("👤 Turn 1: What is 2^10?")
response1 = chat("What is 2 to the power of 10?", config)
print(f"🤖 Claude: {response1}")

print()

# Turn 2 — references the previous answer
print("👤 Turn 2: Now multiply that by 3")
response2 = chat("Now multiply that result by 3.", config)
print(f"🤖 Claude: {response2}")

print()

# Turn 3 — tests long-term memory within the thread
print("👤 Turn 3: What was my first question?")
response3 = chat("What was my very first question in this conversation?", config)
print(f"🤖 Claude: {response3}")

print("\n✅ Claude remembered the entire conversation via checkpointing!")

# 💡 EXPERIMENT: Change the thread_id (e.g., "conversation-002") for Turn 2.
#    Claude will have NO memory of Turn 1 — as if starting fresh.

In [ ]:
# ── Inspecting the Checkpoint State ──────────────────────────────────────────
# You can peek at what's saved in a checkpoint at any time

checkpoint_state = persistent_graph.get_state(config)
messages_in_memory = checkpoint_state.values.get("messages", [])

print(f"📦 Messages stored in checkpoint: {len(messages_in_memory)} total\n")
for i, msg in enumerate(messages_in_memory):
    msg_type = type(msg).__name__
    content_preview = str(msg.content)[:80] if msg.content else "(tool call)"
    print(f"  [{i+1}] {msg_type}: {content_preview}")

---

## 🚀 Part 6: Capstone — Decision-Making Research Agent

Let's put it all together: a **ResearchAgent** that:
1. Takes a research question
2. Plans which tools to use (via Claude)
3. Gathers information using tools in a loop
4. Critiques its own answer (self-critique node)
5. Decides whether to do more research or produce the final report
6. Outputs a structured research report

This is a real pattern used in production agents — the **Plan → Act → Critique → Decide** loop.

In [ ]:
from typing import Optional

# ── Extended State for Research Agent ────────────────────────────────────────
class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    research_question: str
    gathered_facts: list[str]      # facts collected via tools
    critique: str                   # self-critique of current draft
    needs_more_research: bool       # should we keep going?
    iteration: int                  # how many research cycles we've done
    final_report: str               # the output

# LLM for this agent — slightly more capable for quality output
research_llm = ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=800
).bind_tools([calculate, get_weather, lookup_fact])

critique_llm = ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=300
)

print("✅ Research agent state and LLMs ready")

In [ ]:
# ── Research Agent Nodes ──────────────────────────────────────────────────────

def research_node(state: ResearchState) -> dict:
    """Main research node: Claude gathers information using tools."""
    iteration = state.get("iteration", 0) + 1
    print(f"\n🔬 [Research Node] Iteration {iteration}")
    
    context = ""
    if state.get("gathered_facts"):
        context = f"\n\nFacts gathered so far:\n" + "\n".join(f"- {f}" for f in state["gathered_facts"])
    if state.get("critique"):
        context += f"\n\nPrevious critique: {state['critique']}"
    
    prompt = f"""Research question: {state['research_question']}
{context}

Use the available tools to gather more relevant information. 
Focus on what's still missing based on the critique above."""
    
    response = research_llm.invoke([HumanMessage(content=prompt)])
    return {"messages": [response], "iteration": iteration}


def tool_execution_node(state: ResearchState) -> dict:
    """Execute tool calls and collect the results as gathered facts."""
    # Let LangGraph's ToolNode handle the actual tool execution
    tool_results = ToolNode([calculate, get_weather, lookup_fact]).invoke(state)
    
    # Extract facts from tool result messages
    new_facts = []
    for msg in tool_results.get("messages", []):
        if hasattr(msg, "content") and msg.content:
            new_facts.append(str(msg.content))
            print(f"   📦 Tool result: {str(msg.content)[:80]}")
    
    existing_facts = state.get("gathered_facts", [])
    return {
        "messages": tool_results.get("messages", []),
        "gathered_facts": existing_facts + new_facts
    }


def critique_node(state: ResearchState) -> dict:
    """Self-critique: assess whether we have enough information."""
    print("🔍 [Critique Node] Evaluating research quality...")
    
    facts_summary = "\n".join(f"- {f}" for f in state.get("gathered_facts", []))
    prompt = f"""Research question: {state['research_question']}

Facts gathered:
{facts_summary if facts_summary else 'None yet'}

Is this sufficient to answer the research question comprehensively?
Answer with: SUFFICIENT or NEEDS_MORE_RESEARCH
Then in one sentence, explain what's missing (if anything)."""
    
    response = critique_llm.invoke([HumanMessage(content=prompt)])
    critique_text = response.content
    needs_more = "NEEDS_MORE_RESEARCH" in critique_text.upper()
    
    print(f"   → {'Needs more research' if needs_more else 'Sufficient — writing report'}")
    return {"critique": critique_text, "needs_more_research": needs_more}


def write_report_node(state: ResearchState) -> dict:
    """Write the final research report from gathered facts."""
    print("✍️  [Report Node] Writing final report...")
    
    facts_summary = "\n".join(f"- {f}" for f in state.get("gathered_facts", []))
    prompt = f"""Write a clear, structured answer to this research question based on the facts below.

Research question: {state['research_question']}

Facts gathered:
{facts_summary}

Write a 3-4 sentence summary that directly answers the question."""
    
    response = critique_llm.invoke([HumanMessage(content=prompt)])
    return {"final_report": response.content}


print("✅ All research agent nodes defined")

In [ ]:
# ── Routing Functions ─────────────────────────────────────────────────────────

MAX_ITERATIONS = 3  # Safety limit — never loop more than this

def route_after_research(state: ResearchState) -> str:
    """After research node: go to tools if tool calls present, else critique."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "execute_tools"
    return "critique"

def route_after_critique(state: ResearchState) -> str:
    """After critique: loop back to research OR write the report."""
    if state.get("needs_more_research") and state.get("iteration", 0) < MAX_ITERATIONS:
        return "more_research"
    return "write_report"


# ── Build the Research Agent Graph ────────────────────────────────────────────
builder = StateGraph(ResearchState)

builder.add_node("research", research_node)
builder.add_node("execute_tools", tool_execution_node)
builder.add_node("critique", critique_node)
builder.add_node("write_report", write_report_node)

builder.set_entry_point("research")

# research → execute_tools (if tool calls) OR critique (if done)
builder.add_conditional_edges(
    "research",
    route_after_research,
    {"execute_tools": "execute_tools", "critique": "critique"}
)

# After tools, always go back to research
builder.add_edge("execute_tools", "research")

# After critique: loop or finish
builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"more_research": "research", "write_report": "write_report"}
)

# write_report ends the graph
builder.add_edge("write_report", END)

research_agent = builder.compile()
print("✅ Research agent graph compiled!")

In [ ]:
# ── Run the Research Agent ────────────────────────────────────────────────────
print("🚀 Launching research agent...")
print("═" * 60)

research_question = (
    "I'm planning a trip and need to know: what's the weather in Tokyo, "
    "what is Anthropic (the company I'll be meeting), and how much is 2500 "
    "Japanese yen × 0.0067 in USD?"
)

print(f"❓ Research Question: {research_question}")
print("═" * 60)

result = research_agent.invoke({
    "messages": [HumanMessage(content=research_question)],
    "research_question": research_question,
    "gathered_facts": [],
    "critique": "",
    "needs_more_research": True,
    "iteration": 0,
    "final_report": ""
})

print("\n" + "═" * 60)
print("📋 FINAL RESEARCH REPORT")
print("═" * 60)
print(result["final_report"])

print(f"\n📊 Stats:")
print(f"  • Iterations run: {result['iteration']}")
print(f"  • Facts gathered: {len(result['gathered_facts'])}")
print(f"  • Messages in history: {len(result['messages'])}")

# 💡 EXPERIMENT: Change the research_question to something that can't be answered
#    with the 3 tools (e.g., ask about stock prices). Watch the agent hit MAX_ITERATIONS.

---

## 🧩 Part 7: Key LangGraph Patterns — Quick Reference

You've now seen the core patterns. Here they are summarised for quick reference:

In [ ]:
patterns = {
    "Pattern": [
        "Simple Chain",
        "Tool Loop (ReAct)",
        "Router / Dispatcher",
        "Human-in-the-loop",
        "Map-Reduce (parallel)",
        "Plan-and-Execute"
    ],
    "Description": [
        "A → B → C → END. Sequential, no cycles.",
        "LLM → (tool?) → tool_node → LLM → ... → END. Classic ReAct.",
        "classifier → conditional_edge → specialist_1 | specialist_2 | ...",
        "Pause graph at a node using interrupt_before, let human approve, then resume.",
        "Fan-out: split work across N nodes in parallel, then merge results.",
        "Planner node creates a task list, executor node runs tasks one by one."
    ],
    "When to use": [
        "Linear pipelines, data transformation",
        "Any agent that uses tools",
        "Different question types, multi-domain agents",
        "Sensitive actions (send email, make API call)",
        "Independent subtasks that don't depend on each other",
        "Complex multi-step tasks, coding agents"
    ]
}

# Pretty print the table
print(f"{'Pattern':<25} {'Description':<55} {'When to use'}")
print("-" * 120)
for i in range(len(patterns["Pattern"])):
    p = patterns["Pattern"][i]
    d = patterns["Description"][i]
    w = patterns["When to use"][i]
    print(f"{p:<25} {d:<55} {w}")

---

## ✅ Lesson Summary

| Concept | What you learned |
|---------|------------------|
| **StateGraph** | The core LangGraph builder — define state, add nodes, compile |
| **State + Reducers** | TypedDict flows through graph; `add_messages` appends instead of overwriting |
| **Nodes** | Plain Python functions: receive State, return dict of updates |
| **Static Edges** | `add_edge(A, B)` — always goes from A to B |
| **Conditional Edges** | `add_conditional_edges(A, fn, mapping)` — routing function decides next node |
| **Cycles** | `add_edge(tools, agent)` creates a loop — LangGraph handles safely |
| **ToolNode** | LangGraph's built-in node for executing LangChain tools |
| **MemorySaver** | In-memory checkpointer — saves state after each node |
| **thread_id** | Namespaces checkpoints — same ID = same conversation memory |
| **Production upgrade** | Swap MemorySaver → SqliteSaver/PostgresSaver for persistence across restarts |

---

## 🔮 What's Next

**Lesson 12: Fine-tuning Fundamentals** — When does fine-tuning beat prompting? How is training data structured? What is LoRA and why does it make fine-tuning accessible on consumer hardware? We'll also look at when *not* to fine-tune (hint: most of the time).

---

## 🏋️ Homework Challenges

1. **Easy:** Add a 4th tool to the ReAct agent (e.g., `convert_currency`). Test that Claude uses it correctly.

2. **Medium:** Add a `step_count` field to AgentState and a node that refuses to continue if `step_count > 5` (safety guard). Update State and the conditional edge.

3. **Hard:** Implement the **Human-in-the-loop** pattern: add an `interrupt_before=["tools"]` to the graph compilation, so the user can review tool calls before they execute. Docs hint: `graph.compile(checkpointer=memory, interrupt_before=["tools"])`

4. **Open-source prep:** Port your Lesson 9 AutoResearcher agent (plain Python) into a LangGraph graph. This is the architecture pattern you'll use for your open-source project.

In [ ]:
# ── 🎁 Bonus: Visualise Graph Structure ──────────────────────────────────────
# If you're running locally with graphviz installed, this draws the graph
# In Colab, it prints the adjacency structure

print("📐 Research Agent Graph Structure:")
print()

# Print nodes
print("NODES:")
for node in research_agent.nodes:
    print(f"  ● {node}")

print()
print("EDGES (adjacency):")
try:
    # Attempt to get edge info from graph
    graph_dict = research_agent.get_graph().to_json()
    import json
    g = json.loads(graph_dict)
    for edge in g.get("edges", []):
        print(f"  {edge.get('source', '?')} → {edge.get('target', '?')} ", end="")
        if edge.get("conditional"):
            print("(conditional)")
        else:
            print()
except Exception:
    # Fallback: manual description
    edges = [
        ("START", "research", False),
        ("research", "execute_tools", True),
        ("research", "critique", True),
        ("execute_tools", "research", False),
        ("critique", "research", True),
        ("critique", "write_report", True),
        ("write_report", "END", False),
    ]
    for src, dst, cond in edges:
        label = " (conditional)" if cond else ""
        print(f"  {src} → {dst}{label}")

print()
print("🎉 Lesson 11 complete! You can now build stateful, cyclical, checkpointed agent graphs.")